# Phase 2: Validation split and baseline

Goal: set up a time-based validation split, train a simple baseline, and make the first submission.
The baseline is the reference point that every later change is compared against.

Decisions made for this baseline:

- **Validation split:** train on time steps 1 to 21, validate on time steps 22 to 35. The validation
  period is 14 steps, the same length as the test period (36 to 49), and it comes after the training
  period, so it mimics predicting the future.
- **Model:** LightGBM with library default settings, on the 165 raw features only.
- **Class imbalance:** no weighting yet. The metric is ROC-AUC, which only depends on how the
  predictions are ranked, so this gives a clean reference to test weighting against later.
- **Submission model:** after measuring the validation AUC, retrain with the same settings on all
  labeled rows (steps 1 to 35) so the model sees the steps closest to the test period.

Pipeline order: load → features → split → train → validate → predict → save CSV.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42            # fixed seed so results are reproducible
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("train:", train.shape, " test:", test.shape)

train: (141772, 168)  test: (15329, 168)


## 2. Features

The baseline uses only the 165 raw feature columns. `txId` is an ID and `time_step` is left out
because the test time steps (36 to 49) never appear in training, so the model could not learn
anything useful from their values.

Only labeled rows can be used for supervised training, so unlabeled rows (label = NaN) are dropped here.

In [2]:
feature_cols = [c for c in train.columns if c.startswith("feat_")]
print("Number of features:", len(feature_cols))

# Keep only rows that have a label (1 = illicit, 0 = licit)
labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)
print("Labeled rows:", len(labeled), " illicit rate:", round(labeled["label"].mean(), 4))

Number of features: 165
Labeled rows: 31235  illicit rate: 0.1167


## 3. Time-based split

In [ ]:
TRAIN_LAST_STEP = 21   # train on steps 1-21, validate on steps 22-35

tr = labeled[labeled["time_step"] <= TRAIN_LAST_STEP]
va = labeled[labeled["time_step"] > TRAIN_LAST_STEP]

X_tr, y_tr = tr[feature_cols], tr["label"]
X_va, y_va = va[feature_cols], va["label"]

print("Train: steps", tr["time_step"].min(), "to", tr["time_step"].max(),
      "| rows:", len(tr), "| illicit:", y_tr.sum(), "| illicit rate:", round(y_tr.mean(), 4))
print("Valid: steps", va["time_step"].min(), "to", va["time_step"].max(),
      "| rows:", len(va), "| illicit:", y_va.sum(), "| illicit rate:", round(y_va.mean(), 4))

Train: steps 1 to 21 | rows: 19530 | illicit: 1871 | illicit rate: 0.0958
Valid: steps 22 to 35 | rows: 11705 | illicit: 1773 | illicit rate: 0.1515


## 4. Train the baseline

In [4]:
# LightGBM with default settings (100 trees, learning rate 0.1, 31 leaves).
# random_state fixes the seed; verbose=-1 hides the training log.
model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
model.fit(X_tr, y_tr)

,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


## 5. Validate

`predict_proba` returns two columns: the probability of class 0 and of class 1. Column 1 is the
probability of illicit.

In [ ]:
va_pred = model.predict_proba(X_va)[:, 1]
val_auc = roc_auc_score(y_va, va_pred)

# AUC on the training rows to see how large the gap to validation is
tr_auc = roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1])

print("Train AUC:     ", round(tr_auc, 4))
print("Validation AUC:", round(val_auc, 4))

Train AUC:      1.0
Validation AUC: 0.9148


In [6]:
# AUC for each validation time step separately, to see whether performance drifts over time
va_result = va[["time_step", "label"]].copy()
va_result["pred"] = va_pred

rows = []
for step, group in va_result.groupby("time_step"):
    rows.append({
        "time_step": step,
        "n_rows": len(group),
        "n_illicit": group["label"].sum(),
        "auc": roc_auc_score(group["label"], group["pred"]),
    })
per_step_auc = pd.DataFrame(rows).set_index("time_step")
per_step_auc.round(4)

,n_rows,n_illicit,auc
time_step,,,
22,1763,158,0.9991
23,1187,53,0.9591
24,1126,137,0.7866
25,594,118,0.9997
26,517,96,0.9312
27,206,24,0.9995
28,284,85,1.0000
29,1174,329,0.7562
30,524,83,0.9886


In [ ]:
# Check which features did the model rely on most (importance = number of times a feature is used to split)
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance.head(15)

feat_2      298
feat_53     120
feat_58      81
feat_138     69
feat_52      65
feat_16      61
feat_8       60
feat_142     55
feat_88      55
feat_5       50
feat_3       48
feat_89      46
feat_55      44
feat_100     43
feat_90      43
dtype: int32

## 6. Retrain on all labeled rows and predict the test set

Same settings as above, now fitted on steps 1 to 35. The validation AUC above is the estimate of how
this model will do. There is no held-out data left to score this final model on.

In [8]:
final_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
final_model.fit(labeled[feature_cols], labeled["label"])

test_pred = final_model.predict_proba(test[feature_cols])[:, 1]
print("Test predictions: min", round(test_pred.min(), 4), " mean", round(test_pred.mean(), 4),
      " max", round(test_pred.max(), 4))

Test predictions: min 0.0  mean 0.0442  max 0.9997


## 7. Save the submission

In [ ]:
submission = pd.DataFrame({"index": test["index"], "target": test_pred})

assert len(submission) == 15329, "wrong number of rows"
assert list(submission.columns) == ["index", "target"], "wrong columns"
assert submission["target"].notna().all(), "NaN in predictions"
assert (submission["index"].values == sample_sub["index"].values).all(), "index does not match sample submission"

SUB_PATH = "../submissions/sub01_baseline.csv"
submission.to_csv(SUB_PATH, index=False)
print("Saved", SUB_PATH)
submission.head()

Saved ../submissions/sub01_baseline.csv


,index,target
0,0,0.000269
1,1,0.001008
2,2,0.000272
3,3,0.000311
4,4,0.000091


## Results

- **Validation AUC (train 1 to 21, validate 22 to 35):** 0.9148
- **Train AUC and gap to validation:** Train AUC is 1.0, so the model separates the training rows
  perfectly and the gap to validation is about 0.085. The default model is overfitting the training period.
- **Per-step AUC:** Performance is very uneven across the validation steps. Ten of the
  14 steps score above 0.93, and eight of those are above 0.98. Four steps are much weaker: step 29 (0.756),
  step 24 (0.787), step 31 (0.789) and step 32 (0.841). These four steps contain 914 of the 1,773 illicit
  rows in validation, so they pull the overall AUC down. The weak steps are not simply the latest ones
  (steps 33 to 35 all score above 0.99), so this is not a steady decline over time.
- **Feature importance:** `feat_2` is used for far more splits than any other feature (298, against 120
  for the next one, `feat_53`).
- **Course baseline on the public leaderboard:** 0.92119
- **Public leaderboard AUC for `sub01_baseline.csv`:** 0.94660, which is 0.0254 above the course baseline.
- **Does validation agree with the leaderboard?** Only roughly. The public score is 0.032 higher than
  the validation AUC (0.9466 against 0.9148), so validation is pessimistic. Possible reasons: the
  submitted model was retrained on steps 1 to 35 and so saw 60% more labeled rows and more recent data;
  the validation AUC is pulled down by four weak steps (24, 29, 31, 32) that the test period may not
  resemble; and the public leaderboard uses only 30% of the test set. With one submission it is not yet
  known whether changes that improve validation also improve the leaderboard.
